In [13]:
from pathlib import Path

import pandas as pd
import numpy as np
import warnings
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, recall_score,
                              precision_score, classification_report)
import xgboost as xgb
import lightgbm as lgb

In [14]:
INPUT_PATH  = r"Membership_v2_description.xlsx"
OUTPUT_PATH = r"Membership_v2(all_model_results).xlsx"

In [15]:
df = pd.read_excel(INPUT_PATH)

required_columns = [
    "concurrent_streams", "is_user_verified", "gender", "age",
    "duration_days", "is_promotional_price", "is_night_signup",
    "reg_weekday", "amt_7900", "amt_10900", "amt_13900",
    "has_watch_history", "total_watch_count", "watch_days_count",
    "recency", "weekday_watch_ratio", "binge_score", "completion_rate",
    "plan_tier", "currency_type", "payment_device",
    "is_churn_prevented", "repurchase"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 19, 29, 39, 49, 59, np.inf],
    labels=False,
    include_lowest=True,
).fillna(-1).astype(int)

df["plan_tier_enc"] = (
    df["plan_tier"]
    .astype(str)
    .str.strip()
    .map({"기타": 0, "basic": 1, "standard": 2, "premium": 3})
    .fillna(0)
    .astype(int)
)
df["currency_enc"] = (df["currency_type"].astype(str).str.upper() == "USD").astype(int)

device_dummies = pd.get_dummies(
    df["payment_device"].fillna("unknown"),
    prefix="dev",
    drop_first=False,
)
df = pd.concat([df, device_dummies], axis=1)

In [16]:
BASE_FEATURES = [
    "concurrent_streams", "is_user_verified", "gender", "age",
    "duration_days", "is_promotional_price", "is_night_signup",
    "reg_weekday", "amt_7900", "amt_10900", "amt_13900",
    "age_group", "has_watch_history", "total_watch_count",
    "watch_days_count", "recency", "weekday_watch_ratio",
    "binge_score", "completion_rate",
    "plan_tier_enc", "currency_enc"
] + list(device_dummies.columns)

CASE_A = BASE_FEATURES + ["is_churn_prevented"]  # is_churn_prevented 포함
CASE_B = BASE_FEATURES                            # is_churn_prevented 제외

TARGET = "repurchase"
y = df[TARGET]
class_counts = y.value_counts()
if not {0, 1}.issubset(set(class_counts.index)):
    raise ValueError("'repurchase' column must be encoded as 0/1.")
spw = float(class_counts[0] / class_counts[1])  # 클래스 불균형 보정값

In [17]:
RANDOM_STATE = 42
TEST_SIZE    = 0.2
SKF          = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [18]:
def get_default_models(spw):
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "DecisionTree": DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=100, random_state=RANDOM_STATE
        ),
        "XGBoost": xgb.XGBClassifier(
            scale_pos_weight=spw, eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0
        ),
        "LightGBM": lgb.LGBMClassifier(
            class_weight="balanced", random_state=RANDOM_STATE, verbose=-1
        ),
    }

In [19]:
def evaluate(model, X_te, y_te):
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        "AUC":       roc_auc_score(y_te, prob),
        "F1":        f1_score(y_te, pred),
        "Recall":    recall_score(y_te, pred),
        "Precision": precision_score(y_te, pred),
        "Accuracy":  (pred == y_te).mean(),
    }

In [20]:
# ── Step 1. 기본 파라미터 전체 모델 비교 ──────────────────
print("=" * 72)
print("  Step 1. 기본 파라미터 비교 (test set)")
print("=" * 72)
print(f"  {'모델':<22} {'Case':<7} {'AUC':>8} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print("=" * 72)

step1_rows = []
for case_name, features in [("A", CASE_A), ("B", CASE_B)]:
    X      = df[features].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns)
    X_te_s = pd.DataFrame(scaler.transform(X_te),     columns=X_te.columns)

    for mname, model in get_default_models(spw).items():
        Xi_tr = X_tr_s if mname == "LogisticRegression" else X_tr
        Xi_te = X_te_s if mname == "LogisticRegression" else X_te
        model.fit(Xi_tr, y_tr)
        m = evaluate(model, Xi_te, y_te)
        row = {"Case": case_name, "Model": mname, "Tuned": False, **m}
        step1_rows.append(row)
        print(f"  {mname:<22} Case{case_name:<6} "
              f"{m['AUC']:>8.4f} {m['F1']:>7.4f} "
              f"{m['Recall']:>8.4f} {m['Precision']:>8.4f}")

print("=" * 72)

  Step 1. 기본 파라미터 비교 (test set)
  모델                     Case         AUC      F1   Recall     Prec
  LogisticRegression     CaseA        0.6308  0.6767   0.6283   0.7330
  DecisionTree           CaseA        0.5459  0.6906   0.6861   0.6951
  RandomForest           CaseA        0.6168  0.7771   0.8778   0.6971
  GradientBoosting       CaseA        0.6485  0.8056   0.9511   0.6988
  XGBoost                CaseA        0.6256  0.7054   0.6806   0.7321


  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

  LightGBM               CaseA        0.6437  0.7018   0.6671   0.7404
  LogisticRegression     CaseB        0.6290  0.6742   0.6287   0.7267
  DecisionTree           CaseB        0.5372  0.6924   0.6945   0.6904
  RandomForest           CaseB        0.6079  0.7774   0.8795   0.6966
  GradientBoosting       CaseB        0.6431  0.8039   0.9553   0.6939
  XGBoost                CaseB        0.6062  0.6958   0.6717   0.7216
  LightGBM               CaseB        0.6399  0.7005   0.6667   0.7379


In [21]:
# ── Step 2. 장시간 Optuna 하이퍼파라미터 최적화 ──────────────────
# 약 30분 안팎으로 오래 돌려서 AUC 기준 최적 조합을 더 깊게 탐색
MAX_TRIALS_PER_MODEL = 200
TIMEOUT_BY_MODEL = {
    "XGBoost": 300,
    "LightGBM": 300,
    "GradientBoosting": 180,
}

print(f"\n{'='*72}")
print("  Step 2. 장시간 Optuna 최적화")
print(f"{'='*72}")
print(f"  최대 trial 수: {MAX_TRIALS_PER_MODEL}")
print(f"  시간 예산(초): {TIMEOUT_BY_MODEL}")

def cross_validated_auc(trial, build_model, X_data, y_data):
    fold_scores = []
    for fold_idx, (tr_idx, va_idx) in enumerate(SKF.split(X_data, y_data), start=1):
        X_fold_tr = X_data.iloc[tr_idx]
        X_fold_va = X_data.iloc[va_idx]
        y_fold_tr = y_data.iloc[tr_idx]
        y_fold_va = y_data.iloc[va_idx]

        model = build_model(trial)
        model.fit(X_fold_tr, y_fold_tr)
        fold_prob = model.predict_proba(X_fold_va)[:, 1]
        fold_auc = roc_auc_score(y_fold_va, fold_prob)
        fold_scores.append(fold_auc)

        running_auc = float(np.mean(fold_scores))
        trial.report(running_auc, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

def make_sampler():
    return optuna.samplers.TPESampler(
        seed=RANDOM_STATE,
        n_startup_trials=20,
        multivariate=True,
    )

def make_pruner():
    return optuna.pruners.MedianPruner(
        n_startup_trials=20,
        n_warmup_steps=2,
        interval_steps=1,
    )

step2_rows = []

for case_name, features in [("A", CASE_A), ("B", CASE_B)]:
    X = df[features].fillna(0)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    print(f"\n  [Case {case_name}]")

    def build_xgb(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", 200, 1200),
            max_depth=trial.suggest_int("max_depth", 2, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 15),
            gamma=trial.suggest_float("gamma", 0.0, 10.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            scale_pos_weight=spw,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbosity=0,
        )
        return xgb.XGBClassifier(**params)

    study_xgb = optuna.create_study(
        direction="maximize",
        sampler=make_sampler(),
        pruner=make_pruner(),
    )
    study_xgb.optimize(
        lambda trial: cross_validated_auc(trial, build_xgb, X_tr, y_tr),
        n_trials=MAX_TRIALS_PER_MODEL,
        timeout=TIMEOUT_BY_MODEL["XGBoost"],
        gc_after_trial=True,
        show_progress_bar=False,
    )
    best_xgb = build_xgb(study_xgb.best_trial)
    best_xgb.fit(X_tr, y_tr)
    m = evaluate(best_xgb, X_te, y_te)
    row = {
        "Case": case_name,
        "Model": "XGBoost(Optuna)",
        "Tuned": True,
        "CV_AUC": study_xgb.best_value,
        "Best_Params": str(study_xgb.best_trial.params),
        "Completed_Trials": len(study_xgb.trials),
        "Timeout_Seconds": TIMEOUT_BY_MODEL["XGBoost"],
        **m,
    }
    step2_rows.append(row)
    print(f"    XGBoost      CV_AUC={study_xgb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Trials={len(study_xgb.trials)}")

    def build_lgb(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", 200, 1500),
            max_depth=trial.suggest_int("max_depth", 3, 12),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
            num_leaves=trial.suggest_int("num_leaves", 16, 255),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_samples=trial.suggest_int("min_child_samples", 10, 100),
            min_split_gain=trial.suggest_float("min_split_gain", 0.0, 5.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
            class_weight="balanced",
            objective="binary",
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=-1,
        )
        return lgb.LGBMClassifier(**params)

    study_lgb = optuna.create_study(
        direction="maximize",
        sampler=make_sampler(),
        pruner=make_pruner(),
    )
    study_lgb.optimize(
        lambda trial: cross_validated_auc(trial, build_lgb, X_tr, y_tr),
        n_trials=MAX_TRIALS_PER_MODEL,
        timeout=TIMEOUT_BY_MODEL["LightGBM"],
        gc_after_trial=True,
        show_progress_bar=False,
    )
    best_lgb = build_lgb(study_lgb.best_trial)
    best_lgb.fit(X_tr, y_tr)
    m = evaluate(best_lgb, X_te, y_te)
    row = {
        "Case": case_name,
        "Model": "LightGBM(Optuna)",
        "Tuned": True,
        "CV_AUC": study_lgb.best_value,
        "Best_Params": str(study_lgb.best_trial.params),
        "Completed_Trials": len(study_lgb.trials),
        "Timeout_Seconds": TIMEOUT_BY_MODEL["LightGBM"],
        **m,
    }
    step2_rows.append(row)
    print(f"    LightGBM     CV_AUC={study_lgb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Trials={len(study_lgb.trials)}")

    def build_gb(trial):
        params = dict(
            n_estimators=trial.suggest_int("n_estimators", 150, 800),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            min_samples_split=trial.suggest_int("min_samples_split", 10, 100),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 50),
            max_features=trial.suggest_categorical("max_features", [None, "sqrt", "log2"]),
            random_state=RANDOM_STATE,
        )
        return GradientBoostingClassifier(**params)

    study_gb = optuna.create_study(
        direction="maximize",
        sampler=make_sampler(),
        pruner=make_pruner(),
    )
    study_gb.optimize(
        lambda trial: cross_validated_auc(trial, build_gb, X_tr, y_tr),
        n_trials=MAX_TRIALS_PER_MODEL,
        timeout=TIMEOUT_BY_MODEL["GradientBoosting"],
        gc_after_trial=True,
        show_progress_bar=False,
    )
    best_gb = build_gb(study_gb.best_trial)
    best_gb.fit(X_tr, y_tr)
    m = evaluate(best_gb, X_te, y_te)
    row = {
        "Case": case_name,
        "Model": "GradientBoosting(Optuna)",
        "Tuned": True,
        "CV_AUC": study_gb.best_value,
        "Best_Params": str(study_gb.best_trial.params),
        "Completed_Trials": len(study_gb.trials),
        "Timeout_Seconds": TIMEOUT_BY_MODEL["GradientBoosting"],
        **m,
    }
    step2_rows.append(row)
    print(f"    GradBoosting CV_AUC={study_gb.best_value:.4f}  "
          f"Test_AUC={m['AUC']:.4f}  Recall={m['Recall']:.4f}  "
          f"Trials={len(study_gb.trials)}")


  Step 2. 장시간 Optuna 최적화
  최대 trial 수: 200
  시간 예산(초): {'XGBoost': 300, 'LightGBM': 300, 'GradientBoosting': 180}

  [Case A]
    XGBoost      CV_AUC=0.6624  Test_AUC=0.6560  Recall=0.6839  Trials=192
    LightGBM     CV_AUC=0.6617  Test_AUC=0.6552  Recall=0.6823  Trials=200
    GradBoosting CV_AUC=0.6591  Test_AUC=0.6556  Recall=0.9608  Trials=16

  [Case B]
    XGBoost      CV_AUC=0.6547  Test_AUC=0.6498  Recall=0.6721  Trials=200
    LightGBM     CV_AUC=0.6535  Test_AUC=0.6472  Recall=0.6793  Trials=200
    GradBoosting CV_AUC=0.6519  Test_AUC=0.6508  Recall=0.9663  Trials=16


In [22]:
all_rows = step1_rows + step2_rows
res = pd.DataFrame(all_rows)

print(f"\n{'='*72}")
print("  최종 결과 요약 (AUC 내림차순)")
print(f"{'='*72}")
print(f"  {'모델':<28} {'Case':<7} {'AUC':>8} {'F1':>7} {'Recall':>8} {'Prec':>8}")
print(f"{'='*72}")
for case in ["A", "B"]:
    sub = res[res["Case"] == case].sort_values("AUC", ascending=False)
    for _, r in sub.iterrows():
        mark = " ★" if r["AUC"] == sub["AUC"].max() else ""
        print(f"  {r['Model']:<28} Case{r['Case']:<6} "
              f"{r['AUC']:>8.4f} {r['F1']:>7.4f} "
              f"{r['Recall']:>8.4f} {r['Precision']:>8.4f}{mark}")
    print(f"  {'-'*68}")

best_a = res[res["Case"] == "A"].sort_values("AUC", ascending=False).iloc[0]
best_b = res[res["Case"] == "B"].sort_values("AUC", ascending=False).iloc[0]
print(f"\n  CaseA 최고: {best_a['Model']:<28} AUC={best_a['AUC']:.4f}  Recall={best_a['Recall']:.4f}")
print(f"  CaseB 최고: {best_b['Model']:<28} AUC={best_b['AUC']:.4f}  Recall={best_b['Recall']:.4f}")
print(f"  AUC 차이 (A-B): {best_a['AUC'] - best_b['AUC']:+.4f}")

res.to_excel(OUTPUT_PATH, index=False)
print(f"\n저장 완료: {OUTPUT_PATH}")


  최종 결과 요약 (AUC 내림차순)
  모델                           Case         AUC      F1   Recall     Prec
  XGBoost(Optuna)              CaseA        0.6560  0.7143   0.6839   0.7476 ★
  GradientBoosting(Optuna)     CaseA        0.6556  0.8075   0.9608   0.6964
  LightGBM(Optuna)             CaseA        0.6552  0.7121   0.6823   0.7447
  GradientBoosting             CaseA        0.6485  0.8056   0.9511   0.6988
  LightGBM                     CaseA        0.6437  0.7018   0.6671   0.7404
  LogisticRegression           CaseA        0.6308  0.6767   0.6283   0.7330
  XGBoost                      CaseA        0.6256  0.7054   0.6806   0.7321
  RandomForest                 CaseA        0.6168  0.7771   0.8778   0.6971
  DecisionTree                 CaseA        0.5459  0.6906   0.6861   0.6951
  --------------------------------------------------------------------
  GradientBoosting(Optuna)     CaseB        0.6508  0.8073   0.9663   0.6932 ★
  XGBoost(Optuna)              CaseB        0.6498  0.7068